# 0.3 Spectral errors

Runs after **0.2** (clipping). It gives every pixel of every clipped spectrum a flux uncertainty. **The spectra are not smoothed or resampled.** The output has exactly the pixels from 0.2, and the GP later combines them onto its grid (inverse-variance binned in linear flux, then log).

- **Files with errors.** `err_mode` chooses:
  - `"trust"` (default): use them as they are;
  - `"rescale"`: multiply them by this file's estimate/real ratio, with `rescale_shape` either `"constant"` or `"linear"` in λ;
  - `"estimate"`: ignore them.
- **Files without errors** use the estimate.
- **The estimate** follows kaepora's `genivar`: smooth, take |residual|, smooth that, then multiply by √(π/2)/√(1 − 2K₀ + ΣK²). That factor is computed for each spectrum from its own pixel size, so the method works for any instrument. The smoothing here is only used to *measure* the noise; it never changes the flux.

Output: `Inputs/Spectroscopy/<REDUCED_SPEC_SUBDIR>/<SN>/<file>` (currently `1.6_spec_errors`), with columns `wls flux flux_err weight`. `weight = 0` marks pixels clipped in 0.2; their flux is a bridge, not data, and steps 4–6 ignore them. Per-spectrum settings go in `<SN>_error_settings.json`, and there's a list file that step 4 reads.

**How to use it.** Run the **auto-run** first: it saves every spectrum with `DEFAULTS`. Then change any spectrum in its own section below and re-save it. Those saves are marked `customised`, and later auto-runs skip them unless `OVERWRITE_CUSTOMISED = True`.

**Reading the plots:**
- **Top:** the pixels that go downstream, with a ±1σ band. Grey stretches are clipped; orange hatching marks low-S/N flags from 0.2.
- **Middle:** the error estimate (orange), the real σ smoothed (blue), and the σ actually used (black).
- **Bottom:** (flux − smooth)/σ. A robust std of about 1 means the errors match the pixel-to-pixel scatter. Well above 1 means the errors are too small; well below 1, too large. Those are the cases for `"rescale"`.

In [ ]:
import pipeline_config as pconf
pconf.ensure_helper_paths()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spec_errors as se

%matplotlib inline
pd.set_option("display.max_colwidth", 30)

SN = "AT2017gfo"
files = se.list_clipped(SN)          # [(mjd, filename), ...] sorted by MJD
for i, (mjd, f) in enumerate(files):
    s = se.load_clipped(SN, f, mjd)
    print("%2d  %.3f  %-70s %s%s" % (i, mjd, f, "errors" if se.has_errors(s) else "no errors",
                                      "" if se.is_usable(s) else "  (all clipped: skipped)"))

## Default settings

Edit these to change the auto-run. Each spectrum's section starts from a copy.

| key | meaning |
|---|---|
| `err_mode` | for files with errors: `"trust"`, `"rescale"` or `"estimate"` |
| `rescale_shape` | `"constant"` (median estimate/real) or `"linear"` (straight-line fit of the ratio in λ) |
| `est_vexp_smooth` | smoothing whose residuals measure the noise (σ = vexp·λ; kaepora 0.002) |
| `est_vexp_resid` | smoothing of \|residual\| (kaepora 0.015, about 300 Å at 20000 Å; consider lowering it for NIR) |
| `nsig` | smoothing kernels are truncated at ±`nsig` σ |

In [ ]:
DEFAULTS = dict(se.DEFAULTS)
DEFAULTS

---
# Auto-run: every spectrum with `DEFAULTS`

Spectra you have customised below are skipped unless `OVERWRITE_CUSTOMISED = True`.

In [ ]:
SAVE = True
OVERWRITE_CUSTOMISED = False

customised = {n for n, s in se.load_settings(SN).items() if s.get("customised")}
rows = []
for i, (mjd, fname) in enumerate(files):
    spec = se.load_clipped(SN, fname, mjd)
    if not se.is_usable(spec):
        print("%2d skipped (all clipped): %s" % (i, fname))
        continue
    if fname in customised and not OVERWRITE_CUSTOMISED:
        print("%2d kept your customised version: %s" % (i, fname))
        continue
    res = se.reduce_one(spec, DEFAULTS)
    fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
    fig.suptitle("%d: %s" % (i, fname), fontsize=10, y=1.01)
    plt.show()
    if SAVE:
        se.save_errors(SN, res, customised=False)
    rows.append(dict(i=i, **se.summary_row(res)))

In [ ]:
# est/real and chi_std tell you whether to trust or rescale a spectrum's errors:
# chi_std ~ 1 means the errors match the scatter; >1 too small, <1 too large.
pd.DataFrame(rows).set_index("i")

---
# One spectrum at a time

Change `params`, rerun the cell to see the plots, then run the save cell.

---
## 0: `57983.01825_LDSS3_Magellan.dat`
MJD 57983.018, no errors (estimate is used)

In [ ]:
i = 0
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 1: `57983.02788_LDSS3_Magellan.dat`
MJD 57983.028, no errors (estimate is used)

In [ ]:
i = 1
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 2: `57983.0362_LDSS3_Magellan.dat`
MJD 57983.036, no errors (estimate is used)

In [ ]:
i = 2
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 3: `57983.05998_LDSS3_Magellan.dat`
MJD 57983.060, no errors (estimate is used)

In [ ]:
i = 3
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 4: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969001_Phase+1.43d.dat`
MJD 57983.969, has errors

In [ ]:
i = 4
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 5: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969002_Phase+1.43d.dat`
MJD 57983.969, has errors

In [ ]:
i = 5
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 6: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969003_Phase+1.43d.dat`
MJD 57983.969, has errors

In [ ]:
i = 6
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 7: `57983.9914_LDSS3_Magellan.dat`
MJD 57983.991, no errors (estimate is used)

In [ ]:
i = 7
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 8: `57984.0484_LDSS3_Magellan.dat`
MJD 57984.048, no errors (estimate is used)

In [ ]:
i = 8
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 9: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969001_Phase+2.42d.dat`
MJD 57984.969, has errors

In [ ]:
i = 9
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 10: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969002_Phase+2.42d.dat`
MJD 57984.969, has errors

In [ ]:
i = 10
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 11: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57984.969003_Phase+2.42d.dat`
MJD 57984.969, has errors

In [ ]:
i = 11
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 12: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974001_Phase+3.41d.dat`
MJD 57985.974, has errors

In [ ]:
i = 12
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 13: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974002_Phase+3.41d.dat`
MJD 57985.974, has errors

In [ ]:
i = 13
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 14: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57985.974003_Phase+3.41d.dat`
MJD 57985.974, has errors

In [ ]:
i = 14
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 15: `57985.99019_LDSS3_Magellan.dat`
MJD 57985.990, no errors (estimate is used)

In [ ]:
i = 15
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 16: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974001_Phase+4.40d.dat`
MJD 57986.974, has errors

In [ ]:
i = 16
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 17: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974002_Phase+4.40d.dat`
MJD 57986.974, has errors

In [ ]:
i = 17
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 18: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57986.974003_Phase+4.40d.dat`
MJD 57986.974, has errors

In [ ]:
i = 18
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 19: `57987.03511_LDSS3_Magellan.dat`
MJD 57987.035, no errors (estimate is used)

In [ ]:
i = 19
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 20: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980001_Phase+5.40d.dat`
MJD 57987.980, has errors

In [ ]:
i = 20
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 21: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980002_Phase+5.40d.dat`
MJD 57987.980, has errors

In [ ]:
i = 21
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 22: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57987.980003_Phase+5.40d.dat`
MJD 57987.980, has errors

In [ ]:
i = 22
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 23: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990001_Phase+6.40d.dat`
MJD 57988.990, has errors

In [ ]:
i = 23
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 24: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990002_Phase+6.40d.dat`
MJD 57988.990, has errors

In [ ]:
i = 24
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 25: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57988.990003_Phase+6.40d.dat`
MJD 57988.990, has errors

In [ ]:
i = 25
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 26: `57989.98185_LDSS3_Magellan.dat`
MJD 57989.982, no errors (estimate is used)

In [ ]:
i = 26
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 27: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000001_Phase+7.40d.dat`
MJD 57990.000, has errors

In [ ]:
i = 27
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 28: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000002_Phase+7.40d.dat`
MJD 57990.000, has errors

In [ ]:
i = 28
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 29: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57990.000003_Phase+7.40d.dat`
MJD 57990.000, has errors

In [ ]:
i = 29
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 30: `57990.98563_LDSS3_Magellan.dat`
MJD 57990.986, no errors (estimate is used)

In [ ]:
i = 30
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 31: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57991.000001_Phase+8.40d.dat`
MJD 57991.000, has errors

In [ ]:
i = 31
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 32: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57991.000003_Phase+8.40d.dat`
MJD 57991.000, has errors

In [ ]:
i = 32
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 33: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000001_Phase+9.40d.dat`
MJD 57992.000, has errors

In [ ]:
i = 33
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 34: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000002_Phase+9.40d.dat`
MJD 57992.000, has errors

In [ ]:
i = 34
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 35: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57992.000003_Phase+9.40d.dat`
MJD 57992.000, has errors

In [ ]:
i = 35
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 36: `NIR_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000001_Phase+10.40d.dat`
MJD 57993.000, has errors

In [ ]:
i = 36
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## 37: `UVB_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000002_Phase+10.40d.dat`
MJD 57993.000. **Every pixel was clipped in 0.2, so it is skipped.** Go back to 0.2 if you want to keep part of it.

---
## 38: `VIS_AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57993.000003_Phase+10.40d.dat`
MJD 57993.000, has errors

In [ ]:
i = 38
mjd, fname = files[i]
spec = se.load_clipped(SN, fname, mjd)
params = dict(DEFAULTS)
# params["err_mode"] = "rescale"        # "trust" | "rescale" | "estimate"
# params["rescale_shape"] = "linear"    # "constant" | "linear"
# params["est_vexp_resid"] = 0.005      # narrower |residual| smoothing

res = se.reduce_one(spec, params)
fig, axs = se.plot_summary(res, flags=se.low_snr_flags(SN, fname))
# axs["spectrum"].set_xlim(7000, 8000)  # zoom
plt.show()
se.summary_row(res)

In [ ]:
# Run when you're happy with the result above (overwrites the auto-run version).
# se.save_errors(SN, res, customised=True, note="")

---
## What's been saved

In [ ]:
saved = se.load_settings(SN)
if saved:
    table = pd.DataFrame([dict(name=n, **{k: v for k, v in s.items() if k not in ("params", "rescale")})
                          for n, s in saved.items()]).sort_values("mjd").set_index("name")
    print(len(saved), "saved,", int(table["customised"].sum()), "customised")
    display(table)
else:
    print("nothing saved yet")